# CSE 151B — Development notebook

Live in **`notebooks/dev.ipynb`**. Imports resolve **`REPO_ROOT`** automatically so `data/` and `results/` work whether the Jupyter cwd is the project root or `notebooks/`.

This notebook follows **`starter_code_cse151b_comp.ipynb`** end-to-end (vLLM inference → `Judger` scoring → JSONL output), but runs on a **held-out slice** of `public.jsonl` so you can iterate faster.

1. (**Colab**) Copy `public.jsonl` from Drive into `data/` (skip locally if the file already exists).
2. Build `data/dev.jsonl` (stratified MCQ / free-form, reproducible seed).
3. Same prompts, model load, generation loop, and scoring as the starter (`MAX_TOKENS` is 8192 here vs 4096 in starter).
4. Writes **`results/dev_results.jsonl`** (and a matching `.responses.jsonl` checkpoint).

Use the starter notebook for **full** public evaluation; use **`notebooks/submission.ipynb`** for private CSV export.

## 1. Environment (same as starter)

**Colab:** run the `%pip` cell below, restart runtime, continue. **Local:** use `uv` / your venv as in the starter — install the same packages (`vllm`, `transformers`, `bitsandbytes`, …).

In [1]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers "bitsandbytes>=0.48.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 MB 83.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 112.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 87.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 138.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 125.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 183.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 186.4 MB/s eta 0:00:00a 0:00:01
     ━━━━

## 2. Imports & configuration

Matches the starter notebook; dev-specific keys:

- `PUBLIC_PATH` — full `public.jsonl` used only to **build** the dev file
- `DEV_PATH` — subset used as `DATA_PATH` for this run (`data/dev.jsonl`)
- `DEV_FRACTION` — fraction taken from **each** of MCQ and free-form (stratified)
- `DEV_SEED` — reproducible shuffle
- `OUTPUT_PATH` — default `results/dev_results.jsonl`

In [2]:
import json
import os
import random
from pathlib import Path


def _repo_root() -> Path:
    """Project root (parent of `data/`), whether cwd is repo root or `notebooks/`."""
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

# ── Dev split ─────────────────────────────────────────────────────────────────
PUBLIC_PATH   = str(REPO_ROOT / "data/public.jsonl")
DEV_PATH      = str(REPO_ROOT / "data/dev.jsonl")
DEV_FRACTION  = 0.20              # 10% per stratum (MCQ and free-form)
DEV_SEED      = 42
REBUILD_DEV   = True              # Set False to reuse existing DEV_PATH on disk

# ── Experiment knobs ──────────────────────────────────────────────────────────
# PROMPT_VARIANT controls which system prompt set is used.
#   "baseline" — original prompts (matches pub-001 / 8k run)
#   "concise"  — thinking-efficiency prompts (roadmap §1.2)
PROMPT_VARIANT = "concise"

# ── Same as starter ───────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"
DATA_PATH   = DEV_PATH
MAX_TOKENS  = 8192
OUTPUT_PATH = str(REPO_ROOT / f"results/dev_results_{PROMPT_VARIANT}.jsonl")

print(f"REPO_ROOT      = {REPO_ROOT}")
print(f"PROMPT_VARIANT = {PROMPT_VARIANT!r}  →  {OUTPUT_PATH}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import re
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    """Jupyter often replaces stdout with a stream that has no fileno(); vLLM workers need a real FD. Use only around vLLM load/generate so notebook print() still works elsewhere."""
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

REPO_ROOT      = /content
PROMPT_VARIANT = 'concise'  →  /content/results/dev_results_concise.jsonl


## 3. Colab: copy `public.jsonl` from Drive (same as starter)

On **Google Colab**, run this **before** building `dev.jsonl` if the VM has no repo files. Skip locally when `data/public.jsonl` already exists.

In [3]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/public.jsonl`.")
else:
    drive.mount("/content/drive")
    DRIVE_JSONL = Path("/content/drive/MyDrive/CSE151B/data/public.jsonl")
    DRIVE_PROJECT_ROOT = DRIVE_JSONL.parent.parent
    if not DRIVE_JSONL.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_JSONL}\n"
            "Upload public.jsonl or set DRIVE_JSONL."
        )
    (REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)
    dest = REPO_ROOT / "data/public.jsonl"
    shutil.copy2(DRIVE_JSONL, dest)
    print(f"Copied to {dest.resolve()}")

Mounted at /content/drive
Copied to /content/data/public.jsonl


## 4. Build `data/dev.jsonl` (stratified)

Samples `DEV_FRACTION` from MCQ rows and `DEV_FRACTION` from free-form rows separately so the dev mix stays close to public (~33% / ~67%). Sorts by `id` after selection.

In [4]:
def build_dev_jsonl(
    public_path: str,
    dev_path: str,
    fraction: float,
    seed: int,
) -> None:
    with open(public_path) as f:
        all_rows = [json.loads(line) for line in f]
    mcq  = [r for r in all_rows if r.get("options")]
    free = [r for r in all_rows if not r.get("options")]

    rng = random.Random(seed)

    def sample_frac(items: list) -> list:
        if not items:
            return []
        k = max(1, int(len(items) * fraction))
        k = min(k, len(items))
        idx = list(range(len(items)))
        rng.shuffle(idx)
        pick = idx[:k]
        return [items[i] for i in pick]

    dev_mcq  = sample_frac(mcq)
    dev_free = sample_frac(free)
    dev_rows = dev_mcq + dev_free
    dev_rows.sort(key=lambda r: r.get("id", 0))

    Path(dev_path).parent.mkdir(parents=True, exist_ok=True)
    with open(dev_path, "w") as out:
        for row in dev_rows:
            out.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"Wrote {len(dev_rows)} rows to {dev_path}")
    print(f"  MCQ in dev: {len(dev_mcq)} / {len(mcq)}   Free-form in dev: {len(dev_free)} / {len(free)}")
    print(f"  Public total: {len(all_rows)}")


if REBUILD_DEV or not Path(DEV_PATH).is_file():
    build_dev_jsonl(PUBLIC_PATH, DEV_PATH, DEV_FRACTION, DEV_SEED)
else:
    print(f"Using existing {DEV_PATH} (set REBUILD_DEV=True to resample)")

Wrote 225 rows to /content/data/dev.jsonl
  MCQ in dev: 75 / 375   Free-form in dev: 150 / 751
  Public total: 1126


## 5. Load dataset

Uses `DATA_PATH` (`data/dev.jsonl`). Re-run the **build** cell if you change `DEV_FRACTION` / `DEV_SEED`.

In [5]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
print("── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")

Loaded 225 questions  (75 MCQ, 150 free-form) from /content/data/dev.jsonl

── MCQ sample ──
{
  "question": "An ordinary deck of cards containing 26 red cards and 26 black cards is shuffled and dealt out one card at a time without replacement. Let $X_i$ be the color of the $i$th card. Compute $H(X_1,X_2,\\ldots,X_{52})$ in bits.",
  "options": [
    "52",
    "48.8",
    "49.9",
    "50.0",
    "51.5",
    "46.5",
    "50.2",
    "47.3",
    "45.6",
    "53.2"
  ],
  "answer": "B",
  "id": 33
} 

── Free-form sample ──
{
  "question": "Reduce the fraction ${\\frac{25}{40}}$. [ANS]",
  "answer": [
    "5/8"
  ],
  "id": 3
} 



## 6. Prompt construction

Controlled by `PROMPT_VARIANT` in §2:

| Variant | Description | Motivation |
|---------|-------------|------------|
| `"baseline"` | Original pub-001 prompts | Reproducibility baseline |
| `"concise"` | Adds "non-repetitive, commit once identified" instructions | 84% of wrong MCQ are truncated mid-think; traces show circular loops and failure to commit |

Switch variant → re-run this cell + §8 (generation). Results land in `results/dev_results_{PROMPT_VARIANT}.jsonl`.

In [6]:
# ── Baseline prompts (match pub-001 / 8k run) ────────────────────────────────
_MATH_BASELINE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

# ── Concise prompts (roadmap §1.2 — thinking-efficiency experiment) ───────────
# Motivated by data/incorrect_mcq_sample.txt: 84% of wrong MCQ are truncated
# mid-think. Dominant patterns in traces:
#   1. Circular "Wait, no..." loops re-deriving already-correct steps
#   2. Repeated interpretation second-guessing on the same setup
#   3. Model never commits despite having the right answer
# Goal: shorten median think trace so more responses finish within 8k tokens.
_MATH_CONCISE = (
    "You are an expert mathematician. Think step-by-step to solve the problem. "
    "Keep your reasoning focused and non-repetitive: do not re-derive steps you have already "
    "completed, and avoid going in circles. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_CONCISE = (
    "You are an expert mathematician. Think step-by-step to solve the problem. "
    "Keep your reasoning focused and non-repetitive: do not re-derive steps you have already "
    "completed, and avoid going in circles. "
    "Once you have identified the best answer, commit to it immediately and output ONLY its "
    "letter inside \\boxed{}, e.g. \\boxed{C}."
)

# ── Select active prompts based on PROMPT_VARIANT ─────────────────────────────
_PROMPTS = {
    "baseline": (_MATH_BASELINE, _MCQ_BASELINE),
    "concise":  (_MATH_CONCISE,  _MCQ_CONCISE),
}
assert PROMPT_VARIANT in _PROMPTS, f"Unknown PROMPT_VARIANT {PROMPT_VARIANT!r}; choose from {list(_PROMPTS)}"
SYSTEM_PROMPT_MATH, SYSTEM_PROMPT_MCQ = _PROMPTS[PROMPT_VARIANT]


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


print(f"Active variant: {PROMPT_VARIANT!r}")
print(f"\nMCQ system prompt:\n  {SYSTEM_PROMPT_MCQ}\n")
print(f"Math system prompt:\n  {SYSTEM_PROMPT_MATH}\n")

for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

Active variant: 'concise'

MCQ system prompt:
  You are an expert mathematician. Think step-by-step to solve the problem. Keep your reasoning focused and non-repetitive: do not re-derive steps you have already completed, and avoid going in circles. Once you have identified the best answer, commit to it immediately and output ONLY its letter inside \boxed{}, e.g. \boxed{C}.

Math system prompt:
  You are an expert mathematician. Think step-by-step to solve the problem. Keep your reasoning focused and non-repetitive: do not re-derive steps you have already completed, and avoid going in circles. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}.

── MCQ user prompt (first 200 chars) ──
An ordinary deck of cards containing 26 red cards and 26 black cards is shuffled and dealt out one card at a time without replacement. Let $X_i$ be the color of the $i$th card. Compute $H(X_1,X_2,\ldo ...

── F

In [7]:
# SFT pipeline Step 1 — verify Qwen3-Thinking chat template (docs/sft/pipeline.md)
from transformers import AutoTokenizer

_tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

_system, _user = build_prompt(free_sample["question"], free_sample.get("options"))
_reasoning = (
    "We reduce 25/40 by dividing numerator and denominator by 5: "
    "25/5=5 and 40/5=8, so the answer is 5/8."
)
_plain = f"{_reasoning}\n\\boxed{{5/8}}"
_thinking = (
    f"<think>\n{_reasoning}\n</think>\n\\boxed{{5/8}}"
)

_cases = [
    (
        "inference (prompt only)",
        [{"role": "system", "content": _system}, {"role": "user", "content": _user}],
        True,
    ),
    (
        "train: plain assistant",
        [
            {"role": "system", "content": _system},
            {"role": "user", "content": _user},
            {"role": "assistant", "content": _plain},
        ],
        False,
    ),
    (
        "train: <think> assistant",
        [
            {"role": "system", "content": _system},
            {"role": "user", "content": _user},
            {"role": "assistant", "content": _thinking},
        ],
        False,
    ),
]

print(f"MODEL_ID={MODEL_ID}\n")
for label, messages, gen_prompt in _cases:
    text = _tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=gen_prompt
    )
    n_tok = len(_tok.encode(text, add_special_tokens=False))
    print("=" * 72)
    print(f"{label}  ({n_tok} tokens)")
    print("=" * 72)
    print(text)
    print()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

MODEL_ID=Qwen/Qwen3-4B-Thinking-2507

inference (prompt only)  (113 tokens)
<|im_start|>system
You are an expert mathematician. Think step-by-step to solve the problem. Keep your reasoning focused and non-repetitive: do not re-derive steps you have already completed, and avoid going in circles. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}.<|im_end|>
<|im_start|>user
Reduce the fraction ${\frac{25}{40}}$. [ANS]<|im_end|>
<|im_start|>assistant
<think>


train: plain assistant  (166 tokens)
<|im_start|>system
You are an expert mathematician. Think step-by-step to solve the problem. Keep your reasoning focused and non-repetitive: do not re-derive steps you have already completed, and avoid going in circles. Put your final answer inside \boxed{}. If the problem has multiple sub-answers, separate them by commas inside a single \boxed{}, e.g. \boxed{3, 7}.<|im_end|>
<|im_start|>user
Reduce t

## 7. Load model with vLLM (same as starter)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        quantization="bitsandbytes",
        load_format="bitsandbytes",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.88,
        max_model_len=16384,
        trust_remote_code=True,
        max_num_seqs=64,
        max_num_batched_tokens=16384,
    )

sampling_params_free = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)


# ── §1.1 Guided decoding for MCQ ─────────────────────────────────────────────
# vLLM ≥0.20 renamed GuidedDecodingParams → StructuredOutputsParams,
# and SamplingParams kwarg changed from guided_decoding= to structured_outputs=.
# Single-pass: regex (?s:.)*\\boxed{[A-X]} lets the thinking trace be anything,
# then constrains the final tokens to a valid letter. No two-pass fallback.
_GUIDED_AVAILABLE = False
try:
    from vllm.sampling_params import StructuredOutputsParams as _StructuredOutputsParams
    _GUIDED_AVAILABLE = True
except ImportError:
    pass


def make_mcq_sampling_params(n_options: int) -> SamplingParams:
    """Guided SamplingParams for MCQ tail; falls back to free-form params if unavailable."""
    if not _GUIDED_AVAILABLE:
        return sampling_params_free
    last = chr(64 + n_options)
    return SamplingParams(
        max_tokens=MAX_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        structured_outputs=_StructuredOutputsParams(regex=rf"(?s:.)*\\boxed\{{[A-{last}]\}}"),
    )


print(f"Guided decoding available: {_GUIDED_AVAILABLE}")
print("Model loaded.")

INFO 05-23 20:55:01 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.88, 'max_num_batched_tokens': 16384, 'max_num_seqs': 64, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
WARNING 05-23 20:55:01 [arg_utils.py:1467] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 05-23 20:55:20 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-23 20:55:20 [nixl_utils.py:34] NIXL is not available
WARNING 05-23 20:55:20 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-23 20:55:20 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-23 20:55:20 [model.py:1680] Using max model len 16384
INFO 05-23 20:55:20 [schedul

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

INFO 05-23 20:55:21 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None, load_format=bitsandbytes, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=bitsandbytes, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_

model.safetensors.index.json: 0.00B [00:00, ?B/s]

INFO 05-23 20:55:47 [weight_utils.py:615] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 21.965296 seconds
INFO 05-23 20:55:47 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.49 GiB. Available RAM: 79.89 GiB.
INFO 05-23 20:55:47 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-23 20:55:51 [gpu_model_runner.py:4879] Model loading took 2.7 GiB memory and 27.445089 seconds
INFO 05-23 20:56:04 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/95bbff7442/rank_0_0/backbone for vLLM's torch.compile
INFO 05-23 20:56:04 [backends.py:1128] Dynamo bytecode transform time: 12.10 s
INFO 05-23 20:56:14 [backends.py:376] Cache the graph of compile range (1, 16384) for later use
INFO 05-23 20:56:21 [backends.py:391] Compiling a graph for compile range (1, 16384) takes 17.20 s
INFO 05-23 20:56:28 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/364deebc07fb29298a030d0ccc2e51dc97cb384b3e76521da7ac217f400d3e58/rank_0_0/model
INFO 05-23 20:56:28 [monitor.py:53] torch.compile took 36.80 s in total
INFO 05-23 20:56:29 [monitor.py:81] Initial profiling/warmup run took 0.34 s
INFO 05-23 20:56:41 [gpu_model_runner.py:5963] Profiling CUDA graph memory: PIECEWISE=19 (largest=128), FULL=11 (

## 8. Generate responses (same loop as starter)

Checkpoint file: `results/dev_results.responses.jsonl` — delete it to regenerate all dev answers.

In [9]:
CHUNK_SIZE = len(data)

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(OUTPUT_PATH).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / (Path(OUTPUT_PATH).stem + ".responses.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

completed: dict = {}
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    chunk_params = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)
        opts = item.get("options")
        chunk_params.append(
            make_mcq_sampling_params(len(opts)) if opts else sampling_params_free
        )

    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=chunk_params)

    with open(response_checkpoint, "a") as f:
        for item, out in zip(chunk, outputs):
            response = out.outputs[0].text.strip()
            completed[item["id"]] = response
            f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")

Rendering prompts:   0%|          | 0/225 [00:00<?, ?it/s]

Checkpoint path: /content/drive/MyDrive/CSE151B/results/dev_results_concise.responses.jsonl
Generating 225 remaining / 225 total...


Processed prompts:   0%|          | 0/225 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  225/225  (100.0%)
Done. 225 responses ready.


## 9. Score responses (same as starter)

In [10]:
_judger_override = os.environ.get("CSE151B_JUDGER_DIR", "").strip()
try:
    _drive_root = str(DRIVE_PROJECT_ROOT)
except NameError:
    _drive_root = ""
JUDGER_ROOT = _judger_override or _drive_root or str(REPO_ROOT)

print(f"JUDGER_ROOT={JUDGER_ROOT}")
sys.path.insert(0, os.path.abspath(JUDGER_ROOT))
from judger import Judger
judger = Judger(strict_extract=False)

JUDGER_ROOT=/content/drive/MyDrive/CSE151B


In [11]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 225/225 [00:17<00:00, 12.60it/s]

Scoring complete. 225 results.


## 10. Summary

In [12]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("DEV SET RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

DEV SET RESULTS
  MCQ        :   36 /   75  (48.00%)
  Free-form  :   82 /  150  (54.67%)
  Overall    :  118 /  225  (52.44%)


## 11. Save results

Same schema as the starter (`id`, `is_mcq`, `gold`, `response`, `correct`). Uses Drive when `DRIVE_PROJECT_ROOT` exists.

In [13]:
SAVE_EVAL = True

try:
    out_path = DRIVE_PROJECT_ROOT / "results" / Path(OUTPUT_PATH).name
except NameError:
    out_path = Path(OUTPUT_PATH)

out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path.resolve()}")

Saved 225 records to /content/drive/MyDrive/CSE151B/results/dev_results_concise.jsonl


In [14]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")